In [1]:
import torch
import numpy as np
from cryo_sbi.utils.infer_populations import center_models

In [26]:
models_all = torch.load("hsp90_models.pt")

# Select only 4 models
selected_indices = [0, 6, 12, 18]  # example for 20 original states

# Extract only these 4 models
models = models_all[selected_indices, :, :]  # shape [4, 3, 1207]

n_states = len(models)


In [27]:
torch.save(models, "hsp90_models_4.pt")

In [2]:
max_model = torch.load('hsp90_models-small.pt')

In [5]:
# center models (just to be sure)
models = center_models(max_model)
#torch.save(models, "RESNET_18_FFT_Filter/models.pt")
torch.save(models, "RESNET_18/embed_dim_16/models.pt")
torch.save(models, "RESNET_18/embed_dim_64/models.pt")
torch.save(models, "RESNET_18/embed_dim_256/models.pt")

#torch.save(models, "SPATIAL_CRYO/models.pt")
#torch.save(models, "SPATIAL_CRYO_FFT_Filter/models.pt")
#torch.save(models, "SPATIAL_CRYO_GAUSS_FFT_FILTER/models.pt")

In [2]:
def generate_population_weights(n_states, population_steps):
    """
    Generate population weights for mixtures of state 0 and each other state.
    Returns:
        weights: (n_points, n_states)
        pop_fractions: fraction of state 0 for each mixture
        state_pairs: list of tuples (0, i) for each mixture
    """
    fractions = np.linspace(0, 1, population_steps)
    weights_list = []
    pop_fractions = []
    state_pairs = []

    for i in range(1, n_states):
        for f in fractions:
            w = np.zeros(n_states)
            w[0] = f
            w[i] = 1 - f
            weights_list.append(w)
            pop_fractions.append(f)
            state_pairs.append((0, i))

    weights = np.array(weights_list)
    pop_fractions = np.array(pop_fractions)
    state_pairs = np.array(state_pairs)

    weights, idx = np.unique(weights, axis=0, return_index=True)
    pop_fractions = pop_fractions[idx]
    state_pairs = state_pairs[idx]

    return weights, pop_fractions, state_pairs

In [10]:
np.random.dirichlet(np.ones(4), size=1000)

array([[0.11869844, 0.64217876, 0.07479569, 0.16432711],
       [0.1695785 , 0.2673679 , 0.52469027, 0.03836333],
       [0.59824609, 0.00561168, 0.28213304, 0.11400919],
       ...,
       [0.03217713, 0.32378649, 0.03485106, 0.60918532],
       [0.03636011, 0.43498301, 0.3423017 , 0.18635519],
       [0.08612118, 0.48598422, 0.39032141, 0.03757319]], shape=(1000, 4))

In [13]:
# Make initial random weight vector the size of the number of nframes-1
nframes =4 
w_i = np.random.random(nframes-1)
# sort values and insert 0 and 1
w_t = np.append(np.insert(np.sort(w_i), 0, 0.0), 1.0)
# Prepare weight tensor
w_l = np.zeros(nframes)
# Loop through frames & subtract value n from value n+1
for b in range(nframes):
    w_l[b] = w_t[b+1] - w_t[b]
# Convert to tensor
w_l = torch.tensor(w_l, dtype=torch.float)

In [14]:
w_l

tensor([0.2006, 0.4630, 0.2026, 0.1338])

In [29]:

# Generate Dirichlet samples
samples = np.random.dirichlet(np.ones(4), size=1000)

# Round to 3 decimals
samples = np.round(samples, 2)

# Normalize each row so they sum exactly to 1
samples = samples / samples.sum(axis=1, keepdims=True)

print(samples)


[[0.04950495 0.35643564 0.37623762 0.21782178]
 [0.05       0.06       0.2        0.69      ]
 [0.34       0.26       0.07       0.33      ]
 ...
 [0.07       0.33       0.01       0.59      ]
 [0.06060606 0.63636364 0.27272727 0.03030303]
 [0.17       0.15       0.61       0.07      ]]


In [ ]:
def create_weighted_ensemble(w, precision=0.01, max_models=10000, verbose=True):
    """
    Create an ensemble of models by repeating them according to weights.
    Automatically calculates minimum number of repetitions to match weights within specified precision.
    
    Parameters:
        models: torch.Tensor of shape (n_models, ...)
        w: np.array or list of weights (length n_models)
        precision: float, desired precision for weight matching (default: 0.01 for 2 decimals)
        max_models: int, maximum allowed total models (safety limit)
        verbose: bool, whether to print summary
    
    Returns:
        models_ensemble: torch.Tensor of repeated models
        w_actual: np.array of actual weights achieved
        total_models: int, total number of models in ensemble
    """

    # Convert w to numpy array and normalize
    w = np.array(w, dtype=float)
    w = w / w.sum()
    
    # Get non-zero indices
    indices = np.where(w > 0)[0]
    active_weights = w[indices]
    n_active = len(indices)
    
    if n_active == 0:
        raise ValueError("All weights are zero!")
    
    # Special case: only one non-zero weight
    if n_active == 1:
        counts = np.array([1])
        total_models = 1
        w_actual = np.zeros_like(w)
        w_actual[indices[0]] = 1.0
    else:
        # Find minimum multiplier that achieves desired precision
        found = False
        for multiplier in range(1, max_models + 1):
            # Calculate counts
            counts = np.round(active_weights * multiplier).astype(int)
            
            # Skip if any count is 0 (would lose a model)
            if np.any(counts == 0):
                continue
            
            total_models = counts.sum()
            
            # Calculate actual weights from these counts
            actual_weights = counts / total_models
            
            # Check if precision is met for all active weights
            max_error = np.max(np.abs(actual_weights - active_weights))
            
            if max_error < precision:
                found = True
                break
        
        if not found:
            raise ValueError(f"Could not achieve precision {precision} within {max_models} models. "
                           f"Try increasing max_models or relaxing precision.")
        
        # Build full w_actual array
        w_actual = np.zeros_like(w)
        w_actual[indices] = actual_weights

(1000, 4)